# NB_lab_test_trends

Silver → Gold: Lab Test Trends. One row per test type per calendar month with abnormal/critical rates and month-over-month trend analysis.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS

WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date) AS year,
    MONTH(lr.test_date) AS month,
    COUNT(*) AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
  FROM adb_caresync.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

with_rates AS (
  SELECT
    ma.*,
    CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ma.critical_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly_agg ma
),

with_lag AS (
  SELECT
    wr.*,
    LAG(wr.abnormal_rate_pct) OVER (PARTITION BY wr.test_name ORDER BY wr.year, wr.month) AS prev_month_abnormal_rate_pct
  FROM with_rates wr
)

SELECT
  wl.test_name,
  wl.year,
  wl.month,
  c.month_name,
  CONCAT('Q', c.quarter) AS quarter_name,
  wl.total_tests,
  wl.abnormal_test_count,
  wl.critical_test_count,
  wl.abnormal_rate_pct,
  wl.critical_rate_pct,
  wl.prev_month_abnormal_rate_pct,
  CASE
    WHEN wl.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(wl.abnormal_rate_pct - wl.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END AS abnormal_rate_change_pct
FROM with_lag wl
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(wl.year * 10000 + wl.month * 100 + 1 AS INT);

In [0]:
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))